In [1]:
# -*- coding: utf-8 -*-
import os
import json
import numpy as np
from tensorflow import keras
from tqdm import tqdm
from Library import utils, dataset
import logging

# ==============================================================================
# 1. KONFIGURASI PATH
# ==============================================================================
KEY_DATA_DIR = '/Volumes/Extreme SSD/json_indonesia_juli_sesi_5'
KEY_TRAIN_FILE = "extracted_data_3c_sesi_clean.json"

BASE_REP = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mulai_juli/mcquake_ori_file/Code & Figure demo"
MODEL_PATH = os.path.join(BASE_REP, "Pre-trained model/MCU-Quake 5-20")

# Direktori penyimpanan Embedding KDE 3D Domain Indonesia (Standard Key: 'noise' & 'le')
SAVE_EMB_DIR = '/Volumes/Extreme SSD/unduhan_waveform_geofon/output/indonesia_domain_embeddings_3c_le2'
if not os.path.exists(SAVE_EMB_DIR):
    os.makedirs(SAVE_EMB_DIR)

INPUT_WIN = 7 
SAMPLING_RATE = 100
num_points = int(INPUT_WIN * SAMPLING_RATE)

# ==============================================================================
# MAIN EXECUTION
# ==============================================================================
if __name__ == "__main__":
    print("\n" + "="*60)
    print("🚀 FASE ADAPTASI DOMAIN 3C: PEMBUATAN EMBEDDING 32D")
    print("="*60)

    # 1. Load Dataset
    print("[INFO] Memuat Dataset 3C Latih JSON (Domain Indonesia)...")
    train_data = dataset.load_json_data(os.path.join(KEY_DATA_DIR, KEY_TRAIN_FILE))
    keys_list = list(train_data.keys())
    
    # 2. Load & Slice Model (Mendapatkan Feature Extractor 32D)
    print("[INFO] Memuat Pre-trained Model CNN (MCU-Quake)...")
    full_model = keras.models.load_model(filepath=MODEL_PATH, compile=False)
    
    # Mencari layer yang mengeluarkan vektor 32 dimensi
    latent_layer = None
    for layer in full_model.layers:
        # Menangani tuple shape (None, 32)
        if hasattr(layer, 'output_shape') and isinstance(layer.output_shape, tuple):
            if layer.output_shape[-1] == 32:
                latent_layer = layer.output
                break
                
    if latent_layer is None:
        print("⚠️ Peringatan: Layer 32D spesifik tidak ditemukan. Memotong 1 layer klasifikasi terakhir...")
        latent_layer = full_model.layers[-2].output

    # Membangun ulang model yang berhenti di layer 32D
    embedding_model = keras.Model(inputs=full_model.inputs, outputs=latent_layer)
    print(f"✅ Feature Extractor Siap! Dimensi Output: {embedding_model.output_shape}")

    # List penampung fitur laten
    latent_Z_noise, latent_Z_signal = [], []
    latent_N_noise, latent_N_signal = [], []
    latent_E_noise, latent_E_signal = [], []

    # 3. Ekstraksi Fitur Laten 3 Komponen
    print(f"\n[INFO] Mengekstrak fitur laten 3C dari {len(keys_list)} sampel...")
    for i in tqdm(range(len(keys_list)), desc="Ekstraksi CNN 3C"):
        record_key = keys_list[i]
        record = train_data[record_key]
        
        try:
            raw_Zn = np.array(record["Z_noise"][-num_points:])
            raw_Zs = np.array(record["Z"][:num_points])
            
            raw_Nn = np.array(record["N_noise"][-num_points:])
            raw_Ns = np.array(record["N"][:num_points])
            
            raw_En = np.array(record["E_noise"][-num_points:])
            raw_Es = np.array(record["E"][:num_points])

            # Hasil dari model ini dijamin berbentuk (1, 32)
            _in_Zn = utils.latent_codes_1D(raw_Zn, embedding_model)
            _in_Zs = utils.latent_codes_1D(raw_Zs, embedding_model)
            
            _in_Nn = utils.latent_codes_1D(raw_Nn, embedding_model)
            _in_Ns = utils.latent_codes_1D(raw_Ns, embedding_model)
            
            _in_En = utils.latent_codes_1D(raw_En, embedding_model)
            _in_Es = utils.latent_codes_1D(raw_Es, embedding_model)
            
            # .tolist() akan memastikan array [32] tersimpan utuh menjadi list
            latent_Z_noise.append(np.array(_in_Zn).flatten().tolist())
            latent_Z_signal.append(np.array(_in_Zs).flatten().tolist())
            
            latent_N_noise.append(np.array(_in_Nn).flatten().tolist())
            latent_N_signal.append(np.array(_in_Ns).flatten().tolist())
            
            latent_E_noise.append(np.array(_in_En).flatten().tolist())
            latent_E_signal.append(np.array(_in_Es).flatten().tolist())
            
        except Exception as e:
            continue

    # 4. Strukturisasi Format JSON dengan Key 'noise' dan 'le'
    print("\n[INFO] Menyusun struktur kamus data Embedding 3C (Key: 'noise', 'le')...")
    
    embedding_dict_Z = {"noise": latent_Z_noise, "le": latent_Z_signal}
    embedding_dict_N = {"noise": latent_N_noise, "le": latent_N_signal}
    embedding_dict_E = {"noise": latent_E_noise, "le": latent_E_signal}

    # 5. Menyimpan File Embedding 3C Baru
    file_path_Z = os.path.join(SAVE_EMB_DIR, "Embedding data, Z.json")
    file_path_N = os.path.join(SAVE_EMB_DIR, "Embedding data, N.json")
    file_path_E = os.path.join(SAVE_EMB_DIR, "Embedding data, E.json")

    with open(file_path_Z, 'w') as f:
        json.dump(embedding_dict_Z, f)
    with open(file_path_N, 'w') as f:
        json.dump(embedding_dict_N, f)
    with open(file_path_E, 'w') as f:
        json.dump(embedding_dict_E, f)

    print("="*60)
    print("✨ PEMBUATAN EMBEDDING 32D SELESAI!")
    print(f"📁 Direktori Output Baru: {SAVE_EMB_DIR}")
    print("="*60)


🚀 FASE ADAPTASI DOMAIN 3C: PEMBUATAN EMBEDDING 32D
[INFO] Memuat Dataset 3C Latih JSON (Domain Indonesia)...
[INFO] Memuat Pre-trained Model CNN (MCU-Quake)...


2026-08-08 13:12:28.695079: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3 Pro
2026-08-08 13:12:28.695112: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 18.00 GB
2026-08-08 13:12:28.695122: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 6.66 GB
2026-08-08 13:12:28.695154: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-08-08 13:12:28.695169: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


✅ Feature Extractor Siap! Dimensi Output: (None, 32)

[INFO] Mengekstrak fitur laten 3C dari 5159 sampel...


Ekstraksi CNN 3C: 100%|██████████| 5159/5159 [01:40<00:00, 51.50it/s]



[INFO] Menyusun struktur kamus data Embedding 3C (Key: 'noise', 'le')...
✨ PEMBUATAN EMBEDDING 32D SELESAI!
📁 Direktori Output Baru: /Volumes/Extreme SSD/unduhan_waveform_geofon/output/indonesia_domain_embeddings_3c_le2
